In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps03.ipynb")

# PS3 — Regulation Functions and Autoregulation
### BioE 147/247 · Fall 2026

**Out:** Thursday, September 17 · **Due:** Thursday, September 24, 11:59 pm
**Covers:** Sessions 6 and 7 · **43 points** · **Question 6** is required for **BioE 247** (49 total) and is extra credit for **BioE 147** (up to +6)

---

Two sessions, one method. Session 6 counted states to get a regulation function;
session 7 put that function on a pair of axes against removal and read the
circuit off the picture. Questions 1 and 2 are the first; 3, 4 and 5 are the
second.

Four of these questions are the items your handouts said would be here. They
are the same items.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard together,
then write your own solution and your own code. Record who you worked with
below.

**If you used an LLM**, say so briefly and say what for. The conditions are that
you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — limits, orderings,
what happens when a term is switched off. A green visible check means "not
obviously broken", not "right".

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q2"
AI_USE = ""          # e.g. "used an LLM to check my algebra in Q3"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


---
## Question 1 — Two different proteins at *melAB*

*T13, T14. This is item 3 of the session 6 handout.*

MelR$_2^*$ binds the weak proximal operator $O2$ and **activates** — it touches
polymerase, with enhancement factor $f$. CRP$_2^*$ binds the upstream operator
$O1$, **helps recruit MelR**, and does not itself contact polymerase at all.

Two different species, so the two occupancies move independently. Write

$$m = \frac{[\mathrm{MelR}_2^*]}{K_{O2}}, \qquad
  c = \frac{[\mathrm{CRP}_2^*]}{K_{O1}}$$

and let $\omega$ be the cooperative interaction between the two bound proteins,
charged only to the state in which both are on the DNA.

Everything below is in the **weak-promoter limit**, where the fold-change is the
regulation factor and the polymerase weight cancels.

**Q1a.** List the four states, weight them, say what each one fires at, and
turn that into code: `f_melab(m, c, f, omega)` returning $F_{\rm reg}$.

Same convention as item 2 of the session-6 handout, and it is **not** the one on
the slides: this table is already in the weak-promoter limit, so the polymerase
has been divided out, every row has RNAP bound, and the states listed are the
states of **the operators only**. "Nothing bound" therefore fires at 1, not at
0 — it is the unregulated promoter, which is what the others are measured
against.

Write the state list in a comment above the function. The comment is not graded
by the tests, but Q1b assumes you made one.

In [ ]:
def f_melab(m, c, f, omega):
    """Regulation factor for melAB. Weak-promoter limit."""
    ...


print(f"no CRP, saturating MelR : {f_melab(1e6, 0.0, 30.0, 5.0):.2f}")
print(f"with CRP, saturating MelR: {f_melab(1e6, 3.0, 30.0, 5.0):.2f}")

In [ ]:
grader.check("q1a")

<!-- BEGIN QUESTION -->

**Q1b.** *(written)*

1. Take $m \to \infty$ at fixed $c$. Show that $F_{\rm reg} \to f$, and say
   **whose** $f$ that is.
2. There are two proteins on this promoter and only one $f$ in the answer.
   Explain that from the *mechanism*, not from the algebra.
3. What, then, is CRP for? Give the design statement: what does adding the
   upstream site buy you, and what does it not buy you.
4. Your `q1a_omega_one_factorises` test says CRP disappears entirely when
   $\omega = 1$. Say in one sentence why that is the right behaviour and not a
   bug.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 2 — Design an AND-like promoter

*T14. This is item 4 of the session 6 handout.*

Design a promoter whose fold-change is **AND-like** in two inducers: low unless
**both** are present, high when both are.

You choose the architecture. You are not being asked to reproduce a natural
promoter — you are being asked to build one that works, and to say why it
works.

**Q2a.** Implement **your** regulation factor as `f_and(x, y)`, where $x$ and
$y$ are the occupancy variables of your two inducer-controlled proteins (each
$[\,\cdot\,]/K_d$, so both run from 0 upward).

Put any other parameters in as defaults with the values you chose, and say in a
comment what architecture they describe. **The tests call your function at
occupancies of 0 and 10**, so choose parameters for which 10 is a properly
induced state and 0 is properly off. The expression you code **must be the
one you derive in Q2b** — the tests below check behaviour, and a function that
passes them without a state list behind it earns no marks in Q2b.

In [ ]:
def f_and(x, y):          # add your own parameters as keyword defaults
    """YOUR architecture. Say in this docstring what binds where, which
    protein (if any) touches polymerase, and what your parameters mean."""
    ...


for xx, yy in [(0, 0), (10, 0), (0, 10), (10, 10)]:
    print(f"x={xx:3} y={yy:3} -> fold-change {f_and(xx, yy):7.2f}")

In [ ]:
grader.check("q2a")

<!-- BEGIN QUESTION -->

**Q2b.** *(written)*

1. Give the architecture — what binds where, and which protein (if any) touches
   polymerase. A sketch in words is fine.
2. Give the **state list with weights and what each fires at**, and derive the
   regulation factor you coded. Use the session-6 handout's convention: weak
   promoter, operator states only, the unregulated row firing at 1.
3. Give the truth table: the fold-change at (0,0), (high,0), (0,high) and
   (high,high), with your parameter values.
4. **Which knob made it AND-like rather than OR-like?** Name it, and say what
   the promoter would do if you set that knob to 1.
5. What would you have to **measure** to know that the promoter you built
   actually does this? Name the measurement and the failure it would catch.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — The speed-up, and its ceiling

*T15.*

A protein with no degradation tag, in a host dividing every 30 minutes, held at
$p^* = 1000$ molecules:

$$\frac{dp}{dt} = \frac{\alpha}{1 + p/K} - \mu\,p,
\qquad \mu = \frac{\ln 2}{30}, \qquad
\alpha = \alpha_0\left(1 + \frac{p^*}{K}\right)$$

where $\alpha_0 = \mu p^*$ and $\alpha$ is always chosen, as in class, to put the
crossing back at $p^*$. The **repression ratio** is $R = \alpha/\alpha_0 = 1 +
p^*/K$; $R = 1$ is no feedback at all.

**Q3a.** Write `t_half_nar(R, T_d=30.0, p_star=1000.0)` returning the time
in minutes for $p$ to reach $p^*/2$ starting from $p(0) = 0$.

Integrate it. Do not try to solve it in closed form — that is Q3c's job, and
only in a limit.

**Handle $R = 1$.** It is a legitimate input — no feedback — and $K = p^*/(R-1)$
divides by zero there. Decide what the answer is *before* you write the
special case; you already know it from session 5.

In [ ]:
from scipy.optimize import brentq


def t_half_nar(R, T_d=30.0, p_star=1000.0):
    """Time to reach p*/2 under negative autoregulation of repression ratio R."""
    ...


for R in (1.0, 2.0, 4.0, 10.0, 1000.0):
    t = t_half_nar(R)
    print(f"R = {R:7.1f}   t_half = {t:6.2f} min   speed-up = {30.0 / t:.2f}x")

In [ ]:
grader.check("q3a")

**Q3b.** Your Q3a numbers stop moving. Write `speedup_ceiling(T_d=30.0)`
returning the largest speed-up $t_{1/2}(R=1)/t_{1/2}(R)$ that negative
autoregulation can ever give in a host with doubling time `T_d`.

You may find it numerically — take $R$ large — or from the closed form you are
about to derive in Q3c. Either is acceptable, and one of them is much less
work.

**You have already read this number.** The caption of Figure 2 in Rosenfeld *et
al.* states the limiting rise time for strong autorepression, and the figure
itself is this calculation. Their value is rounded; yours does not have to be.
The tests accept either, and Q3c is where exactness is worth marks.

In [ ]:
def speedup_ceiling(T_d=30.0):
    """Largest response-time speed-up negative autoregulation can give."""
    ...


print(f"ceiling = {speedup_ceiling():.3f}x")
print(f"check by brute force: {30.0 / t_half_nar(1e6):.3f}x")

In [ ]:
grader.check("q3b")

<!-- BEGIN QUESTION -->

**Q3c.** *(written)* Now get that ceiling honestly.

1. In the limit of **strong repression** ($K \ll p$ over essentially the whole
   approach), show that the production term collapses to $\alpha_0 p^*/p$, so
   that
   $$\frac{dp}{dt} = \frac{\alpha_0 p^*}{p} - \mu\,p$$
   Say what physical statement that equation is making about the promoter.
2. Solve it with $p(0) = 0$. *(Substitute $u = p^2$; it becomes the same
   first-order linear equation you solved in PS2.)*
3. Read the response time off your solution, in units of the cell cycle, and
   compare it with the number Rosenfeld, Elowitz & Alon quote for their
   autoregulated strain in Figure 3.
4. State the ceiling as a sentence a designer would use — including what it
   says about the 5-fold that paper measures and the 1.8-fold we derived in
   class.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 4 — Positive autoregulation, computationally

*T16. This is item 3 of the session 7 handout.*

Now the protein **activates** its own promoter, with $n$ sites bound
cooperatively and a leaky basal rate:

$$\frac{dp}{dt} = \underbrace{\alpha_{\rm basal}
+ \alpha_{\max}\frac{p^{n}}{K^{n} + p^{n}}}_{\text{production}} - \mu\,p$$

Run the cell below; it defines the parameters every part of this question uses.
They are session 7's.

In [ ]:
MU_30 = np.log(2.0) / 30.0          # /min, 30-minute doubling
ALPHA_0 = MU_30 * 1000.0            # /min, the unregulated rate for p* = 1000

BASAL = 0.14 * ALPHA_0              # /min, the leak
AMAX = 1.6 * ALPHA_0                # /min, the fully activated rate
KHALF = 900.0                       # molecules
PMAX = 2000.0                       # only look for crossings below this

print(f"basal {BASAL:.2f}/min, alpha_max {AMAX:.2f}/min, mu {MU_30:.4f}/min")

**Q4a.** Write `crossings(n, mu=MU_30, pmax=PMAX)` returning a **sorted list**
of every $p$ in $(0, p_{\max}]$ where production equals removal.

Find them properly: scan for sign changes in production minus removal and
bracket a root-finder on each. Do not eyeball a plot, and do not assume how many
there are.

In [ ]:
def production(p, n, basal=BASAL, amax=AMAX, K=KHALF):
    """Production rate (molecules/min) at protein level p."""
    return basal + amax * p ** n / (K ** n + p ** n)


def crossings(n, mu=MU_30, pmax=PMAX, basal=BASAL, amax=AMAX, K=KHALF):
    """Every p in (0, pmax] where production equals removal, sorted."""
    ...


for n in (1, 2, 4):
    print(f"n = {n}: {[round(r) for r in crossings(n)]}")

In [ ]:
grader.check("q4a")

**Q4b.** Write `classify(p, n, mu=MU_30)` returning the string `"stable"` or
`"unstable"` for a crossing at `p`.

Use the argument from session 7 — the **sign of the gap** either side — not
anything you have not been taught yet. A crossing where production falls
*through* removal from above is stable.

In [ ]:
def classify(p, n, mu=MU_30, basal=BASAL, amax=AMAX, K=KHALF, eps=1e-4):
    """'stable' or 'unstable' for the crossing at p, from the sign of the gap."""
    ...


for r in crossings(4.0):
    print(f"p = {r:7.1f}   {classify(r, 4.0)}")

In [ ]:
grader.check("q4b")

**Q4c.** Write `critical_n(mu=MU_30)` returning the cooperativity at which the
two extra crossings appear, to three decimal places.

Bisect on the **number of crossings**. Everything else is fixed.

In [ ]:
def critical_n(mu=MU_30, lo=1.0, hi=6.0, tol=1e-4):
    """The cooperativity at which the second and third crossings appear."""
    ...


nc = critical_n()
print(f"n_c = {nc:.3f}")
print(f"just below: {len(crossings(nc - 0.02))} crossing(s);  "
      f"just above: {len(crossings(nc + 0.02))}")

In [ ]:
grader.check("q4c")

**Q4d.** $\mu$ is the *slope of the removal line*, and it belongs to the host,
not to you. Write `mu_window(n=4.0)` returning `(mu_low, mu_high)`, the range of
$\mu$ over which this circuit has three crossings.

**Watch your search window.** `PMAX = 2000` is the right bound at $\mu =
0.0231$/min, and it is the *wrong* bound at half that: a shallower removal line
meets the production curve much further out. Work out roughly where the highest
crossing can possibly be — production is bounded above — and search past it. A
`crossings` call that silently reports two roots instead of three will give you
a plausible, wrong answer here.

Then run the cell below it, which converts your answer into doubling times.
Look at the number it prints for the fast edge before you go on to Q5.

In [ ]:
def mu_window(n=4.0, lo=1e-4, hi=0.2, tol=1e-7):
    """Range of mu over which the circuit is bistable, as (mu_low, mu_high)."""
    ...


lo, hi = mu_window(4.0)
print(f"bistable for mu in [{lo:.4f}, {hi:.4f}] /min")
print(f"that is doubling times from {np.log(2)/hi:.1f} to {np.log(2)/lo:.1f} min")
print(f"the host in class divides every 30.0 min")

In [ ]:
grader.check("q4d")

---
## Question 5 — The design item

*T15. This is item 4 of the session 7 handout.*

You are handed a protein that must reach $p^* = 1000$ molecules **within 15
minutes** of induction and then hold that level for **a day**. The host divides
every 30 minutes. You may:

- add a degradation tag (Andersen's best is LAA, a 40-minute half-life) and
  raise $\alpha$ to hold the level;
- wire negative autoregulation at a repression ratio of your choosing;
- or both.

**Q5a.** Write `synthesised_in_24h(route, R=2.0)` returning the **total number
of protein molecules synthesised per cell over 24 hours**, starting from
$p(0)=0$, for `route` in `"none"`, `"tag"`, `"nar"`.

Integrate the production rate alongside the protein, not afterwards: add a
second state variable whose derivative is the production rate.

All three routes must hold $p^* = 1000$. `R = 2` is the default only because it
is the design point from class; Q5b will have you choose your own, and the
function should take whatever you pass it.

In [ ]:
def synthesised_in_24h(route, R=2.0, T_d=30.0, p_star=1000.0,
                       tag_half_life=40.0):
    """Total molecules synthesised per cell over 24 h, holding p* throughout."""
    ...


for r in ("none", "tag", "nar"):
    print(f"{r:>5}: {synthesised_in_24h(r):9,.0f} molecules in 24 h")

In [ ]:
grader.check("q5a")

<!-- BEGIN QUESTION -->

**Q5b.** *(written)* Now choose, and defend it in no more than 300 words.

1. **Can the tag meet the specification at all?** Give the number that decides
   it. Then give the smallest repression ratio that does meet it (you have
   `t_half_nar` from Q3a — use it).
2. **Price your choice** over the day, using Q5a, and say which term of the
   bill the "hold it for a day" clause makes decisive — then say what the
   comparison would look like if the clause said twenty minutes instead.
3. Name one thing you would have to check in the lab that none of this
   arithmetic can tell you.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 6 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 6 of your 49
points.

**BioE 147** — optional, worth up to 6 points of extra credit on top of
43. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

<!-- BEGIN QUESTION -->

**Q6.** Q4c found the critical cooperativity by counting crossings and
bisecting. Get it analytically instead.

At the moment the two new crossings appear, the production curve does not merely
touch the removal line — it is **tangent** to it. That is two conditions at the
same point:

$$\alpha_{\rm basal} + \alpha_{\max}\frac{p^{n}}{K^{n}+p^{n}} = \mu p,
\qquad
\frac{d}{dp}\left[\alpha_{\max}\frac{p^{n}}{K^{n}+p^{n}}\right] = \mu$$

1. Write the second condition out. (The derivative of a Hill function is worth
   doing once by hand; you derived the function itself in session 4.)
2. Solve the pair for $(p_c, n_c)$ — numerically is fine, but the **two
   equations must come from the tangency argument**, not from counting roots.
   Report $n_c$ to three decimals and compare with your Q4c answer.
3. This is a **saddle-node bifurcation**. You meet the name in session 9, on the
   day this set is due — so answer from what you can see in Q4, not from a
   definition. Say what is qualitatively different about the way the two new
   crossings appear, compared with a crossing that simply slides along the axis
   as a parameter changes. (Careful: session 8 will introduce a *saddle*, which
   is a kind of fixed point in two dimensions. That is a different object from
   a saddle-**node**, which is an event. Nothing here is two-dimensional.)
4. One sentence for a designer: what does the existence of a sharp $n_c$ mean
   for someone who is trying to *build* a circuit with memory and has only
   approximate control over cooperativity?

_Type your answer here, replacing this text._

<!-- END QUESTION -->

